# Final Integration Notebook

This notebook summarizes the final background, literature alignment, implementation approach, and runnable demo results for the three ARI711S project tasks. The result values are aligned with `reports/evaluation_metrics.json` and `reports/evaluation_notes.md`.

**Important data note:** the official assignment datasets for flights and staff scheduling are not tracked in this repository, so Parts 1 and 2 were validated on generated demo datasets. Part 3 was trained and evaluated on the extracted GTSRB training dataset at `gtsrb/Train`.

## Background And Literature Alignment

The project combines three AI methods: graph search, constraint satisfaction, and supervised image classification. The background text in `reports/background.md` explains the theory in the same order as the final implementation, and `references/literature-summary.md` records the sources used to justify each method.

- The flight task uses BFS because the route graph is unweighted and the objective is the minimum number of flight connections.
- The scheduler task is modeled as a CSP because each weekly shift must be assigned a nurse while satisfying leave, rest, and weekly workload constraints.
- The traffic sign task uses a CNN because image classification requires visual feature extraction from fixed-size image arrays.
- The final evaluation reports accuracy, a confusion matrix, training curves, and sample predictions for the CNN pipeline.

## Part 1 - Flight Connections

**Implementation:** `src/part1_search/flights.py`

The flight network is represented as a directed graph. Cities are states, direct flights are actions, and each edge has equal cost. The implemented `neighbors_for_city(city_id, cities)` function returns directly reachable destination cities, and `shortest_path(source, target, cities)` applies BFS to return the minimum-hop route.

**Demo result:** the evaluation used `data/demo_flights/` and passed `4 / 4` test cases. The example route `Windhoek -> Johannesburg -> Nairobi -> Cairo` has hop count `3`.

**Figure:** `reports/figures/part1_route_lengths.png`

## Part 2 - Hospital Shift Scheduler

**Implementation:** `src/part2_optimization/run_scheduler.py` and `src/models/Csp.py`

The scheduler is a CSP with 21 variables: 7 days multiplied by 3 shifts per day. Each variable's domain is the set of nurses. The solver enforces leave-day unary constraints, Night-to-next-Morning rest constraints, and a maximum of 5 shifts per nurse.

The implemented pipeline runs node consistency, AC-3, MRV-based backtracking, and forward checking. This keeps the search focused on valid assignments and detects dead ends early.

**Demo result:** the evaluation used `data/demo_staff/staff_small.txt`, assigned all 21 shifts, produced `0` leave violations, produced `0` rest-rule violations, and kept the maximum shifts assigned to any nurse at `5`. The fairness standard deviation was `1.6`.

**Figures:** `reports/figures/part2_schedule_overview.png`, `reports/figures/part2_shift_distribution.png`

## Part 3 - Traffic Sign Recognition

**Implementation:** `src/train.py`, `src/part3_ml/train.py`, and `src/models/Cnn.py`

The assignment context is the German Traffic Sign Recognition Benchmark (GTSRB), which contains 43 traffic sign classes. The final workspace evaluation used the extracted dataset at `gtsrb/Train`, which contains 39,209 labeled images across class folders `0` through `42`.

The preprocessing pipeline loads class-numbered image folders, resizes each image to `30x30`, normalizes pixel values to `[0, 1]`, and uses an 80/20 stratified train/test split. The baseline CNN uses two convolutional blocks, max pooling, dropout, a dense hidden layer, and a 43-unit softmax output layer.

**GTSRB result:** the evaluation trained for `6` epochs with batch size `32`. Final held-out test accuracy was `0.9872`, with `7,742 / 7,842` correct predictions. The saved model is `reports/artifacts/gtsrb_model.keras`.

**Figures:** `reports/figures/confusion.png`, `reports/figures/sample_predictions.png`, `reports/figures/training_curve.png`

## Final Report Artifacts

- Background and theory: `reports/background.md`
- Literature summary: `references/literature-summary.md`
- Results summary: `reports/results.md`
- Evaluation notes: `reports/evaluation_notes.md`
- Canonical metrics: `reports/evaluation_metrics.json`
- Slide storyline: `presentation/slide_draft.md`

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# Load evaluation metrics
metrics_path = '../reports/evaluation_metrics.json'
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

print("✓ Loaded evaluation metrics from reports/evaluation_metrics.json")

✓ Loaded evaluation metrics from reports/evaluation_metrics.json


In [2]:
# Part 1 Results Summary
print("=" * 60)
print("PART 1 - FLIGHT CONNECTIONS (BFS Search)")
print("=" * 60)

part1 = metrics['part1']
print(f"\nDataset: {part1['dataset']}")
print(f"Test Cases: {part1['cases_passed']}/{part1['cases_run']} passed ✓")
print(f"\nTest Results:")
print("-" * 60)

for result in part1['results']:
    status = "✓" if result['passed'] else "✗"
    print(f"{status} {result['case']}: {result['hop_count']} hops" if result['path_found'] else f"{status} {result['case']}: No path found")

print(f"\nFigure: {part1['figure']}")
print("Implementation: src/part1_search/flights.py")

PART 1 - FLIGHT CONNECTIONS (BFS Search)

Dataset: data\demo_flights
Test Cases: 4/4 passed ✓

Test Results:
------------------------------------------------------------
✓ Windhoek to Cairo: 3 hops
✓ Johannesburg to Lagos: 1 hops
✓ Nairobi to Nairobi: 0 hops
✓ Cairo to Windhoek: No path found

Figure: reports\figures\part1_route_lengths.png
Implementation: src/part1_search/flights.py


In [ ]:
# Part 2 Results Summary
print("\n" + "=" * 60)
print("PART 2 - HOSPITAL SHIFT SCHEDULING (CSP Solver)")
print("=" * 60)

part2 = metrics['part2']
metrics_p2 = part2['metrics']
print(f"\nDataset: {part2['dataset']}")
print(f"Total Nurses: {part2['nurses']}")
print(f"Total Shifts: 21 (7 days × 3 shifts)")
print(f"\nOptimization Results:")
print(f"  • All Shifts Assigned: {metrics_p2['fully_assigned']} ✓")
print(f"  • Leave Violations: {len(metrics_p2['leave_violations'])} ✓")
print(f"  • Rest Rule Violations: {len(metrics_p2['rest_violations'])} ✓")
print(f"  • Max Shifts per Nurse: {metrics_p2['max_shifts_per_nurse']}")
print(f"  • Fairness (StdDev): {metrics_p2['fairness_stddev']:.1f}")

print(f"\nFigures:")
for fig in part2['figures']:
    print(f"  • {fig}")
print("Implementation: src/part2_optimization/run_scheduler.py, src/models/Csp.py")


PART 2 - HOSPITAL SHIFT SCHEDULING (CSP Solver)

Dataset: data\demo_staff\staff_small.txt
Total Shifts: 21 (7 days × 3 shifts)

Optimization Results:


KeyError: 'shifts_assigned'

In [ ]:
# Part 3 Results Summary
print("\n" + "=" * 60)
print("PART 3 - TRAFFIC SIGN RECOGNITION (CNN Classification)")
print("=" * 60)

part3 = metrics['part3']
print(f"\nDataset: {part3['dataset']}")
print(f"Total Images: {part3['total_images']}")
print(f"Test Samples: {part3['test_images']}")
print(f"Classes: 43 (German Traffic Signs)")

print(f"\nTraining Configuration:")
print(f"  • Epochs: {part3['epochs']}")
print(f"  • Batch Size: {part3['batch_size']}")
print(f"  • Loss Function: Categorical Crossentropy")
print(f"  • Optimizer: Adam")

print(f"\n★ TEST ACCURACY: {part3['accuracy']:.4f} ({part3['correct_predictions']}/{part3['test_images']} correct) ★")
print(f"  • Test Loss: {part3['loss']:.4f}")
print(f"  • Confusion Matrix Shape: {part3['confusion_matrix_validation']['shape']}")

print(f"\nModel Artifact: {part3['model_artifact']}")
print(f"Figures:")
for fig in part3['figures']:
    print(f"  • {fig}")
print("Implementation: src/train.py, src/part3_ml/train.py, src/models/Cnn.py")

In [ ]:
# Display all evaluation figures
print("\n" + "=" * 60)
print("EVALUATION FIGURES")
print("=" * 60)

figures_dir = '../reports/figures'
figure_files = [f for f in os.listdir(figures_dir) if f.endswith('.png')]
figure_files.sort()

print(f"\nGenerated {len(figure_files)} figures:\n")

# Create a 3x2 subplot layout
n_figures = len(figure_files)
n_rows = (n_figures + 2) // 3
n_cols = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for idx, img_file in enumerate(figure_files):
    img_path = os.path.join(figures_dir, img_file)
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(img_file, fontsize=10, fontweight='bold')
    axes[idx].axis('off')
    print(f"  {idx + 1}. {img_file}")

# Hide unused subplots
for idx in range(len(figure_files), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/submission_figures_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Generated submission_figures_grid.png")

In [ ]:
# Final Summary Table
print("\n" + "=" * 60)
print("FINAL PROJECT SUMMARY")
print("=" * 60)

summary_data = {
    'Task': ['Part 1: Flight Search', 'Part 2: Shift Scheduling', 'Part 3: Traffic Signs'],
    'Method': ['BFS Graph Search', 'CSP with AC-3', 'Convolutional Neural Network'],
    'Status': ['✓ Complete', '✓ Complete', '✓ Complete'],
    'Key Result': [
        '4/4 test cases passed',
        'All 21 shifts assigned',
        '98.72% accuracy (7,742/7,842)'
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

print("\n" + "=" * 60)
print("PROJECT COMPLETION STATUS")
print("=" * 60)
print("""
✓ Background theory documented (reports/background.md)
✓ Literature review completed (references/literature-summary.md)
✓ All three parts implemented and tested
✓ Evaluation metrics computed (reports/evaluation_metrics.json)
✓ Figures generated and validated
✓ Model artifact saved (reports/artifacts/gtsrb_model.keras)
✓ Results summary available (reports/results.md)

Ready for final submission!
""")

print("=" * 60)
print("REFERENCE DOCUMENTS")
print("=" * 60)
print(f"""
• Background: reports/background.md
• Literature: references/literature-summary.md
• Results: reports/results.md
• Metrics: reports/evaluation_metrics.json
• Evaluation Notes: reports/evaluation_notes.md
• Presentation Outline: presentation/slide_draft.md
""")